# Chapter 4 Lab: Linear Models and Representation (`ch03`)

In this lab, we explore linear and logistic regression, feature engineering, regularization, and the pitfalls of coefficient interpretation. We will fit models to synthetic and real data, visualize decision boundaries, and learn when linear models succeed and fail.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Seed for reproducibility
np.random.seed(42)

## 1. Linear Regression from Features

We generate synthetic data where y = 3*x₁ - 2*x₂ + noise, then fit a linear regression model and check whether it recovers the true coefficients.

In [ ]:
# Generate synthetic data with known coefficients
n_samples = 200
X = np.random.randn(n_samples, 2)
true_coeff = np.array([3.0, -2.0])
y = X @ true_coeff + np.random.randn(n_samples) * 0.1

# Fit linear regression
model = LinearRegression()
model.fit(X, y)

# Print results
print("True coefficients:", true_coeff)
print("Fitted coefficients:", model.coef_)
print("Fitted intercept:", model.intercept_)
print(f"R² score: {model.score(X, y):.4f}")
print("\nThe model successfully recovers the true underlying relationship.")

## 2. Feature Engineering

We use the make_moons dataset: linear logistic regression fails on this nonlinear data, but adding polynomial features enables the model to fit well. We visualize both decision boundaries.

In [ ]:
# Generate make_moons data
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)

# Create a mesh for decision boundary visualization
def plot_decision_boundary(ax, X, y, model, title, preprocessor=None):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    X_test = np.c_[xx.ravel(), yy.ravel()]
    if preprocessor is not None:
        X_test = preprocessor.transform(X_test)
    Z = model.predict(X_test).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, edgecolors='k', s=50)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

# Fit logistic regression without feature engineering
lr_basic = LogisticRegression(max_iter=1000, random_state=42)
lr_basic.fit(X_moons, y_moons)

# Fit logistic regression with polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_moons_poly = poly.fit_transform(X_moons)
lr_poly = LogisticRegression(max_iter=1000, random_state=42)
lr_poly.fit(X_moons_poly, y_moons)

# Plot both
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_decision_boundary(axes[0], X_moons, y_moons, lr_basic, 'Linear Decision Boundary (Fails)')
plot_decision_boundary(axes[1], X_moons, y_moons, lr_poly, 'Polynomial Features (Degree=2)', poly)
plt.tight_layout()
plt.show()

print(f"Linear model accuracy: {lr_basic.score(X_moons, y_moons):.3f}")
print(f"Polynomial model accuracy: {lr_poly.score(X_moons_poly, y_moons):.3f}")

## 3. Regularization Path

We fit Lasso regression with varying regularization strength (alpha) and plot how coefficients change. This visualizes how regularization induces sparsity.

In [ ]:
# Generate data with many features
n_samples, n_features = 100, 20
X_reg = np.random.randn(n_samples, n_features)
true_coeff_sparse = np.zeros(n_features)
true_coeff_sparse[:3] = [2.0, -1.5, 0.8]
y_reg = X_reg @ true_coeff_sparse + np.random.randn(n_samples) * 0.5

# Standardize features
scaler = StandardScaler()
X_reg_scaled = scaler.fit_transform(X_reg)

# Fit Lasso for a range of alphas
alphas = np.logspace(-3, 1, 20)
coeff_paths = []

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_reg_scaled, y_reg)
    coeff_paths.append(lasso.coef_)

coeff_paths = np.array(coeff_paths)

# Plot coefficient paths
plt.figure(figsize=(10, 6))
for i in range(n_features):
    plt.plot(alphas, coeff_paths[:, i], label=f'Feature {i+1}' if i < 5 else '')
plt.xlabel('Regularization Strength (alpha)')
plt.ylabel('Coefficient Value')
plt.title('Lasso Coefficient Paths: Sparsity Increases with Alpha')
plt.xscale('log')
plt.axhline(0, color='k', linestyle='--', alpha=0.3)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Count non-zero coefficients at each alpha
sparsity = np.sum(np.abs(coeff_paths) > 1e-4, axis=1)
print("Non-zero coefficients at each alpha:")
for alpha, n_nonzero in zip(alphas, sparsity):
    print(f"  alpha={alpha:.4f}: {n_nonzero} non-zero coefficients")

## 4. Coefficient Interpretation Pitfalls

When features are correlated, regression coefficients become unstable and unreliable for interpretation. We demonstrate this instability and show how Ridge regularization stabilizes them.

In [ ]:
# Generate highly correlated features (r ≈ 0.95)
n_samples = 100
x1 = np.random.randn(n_samples)
x2 = x1 + 0.1 * np.random.randn(n_samples)  # x2 ≈ x1
X_corr = np.column_stack([x1, x2])

# True relationship: y = x1, but x2 is almost redundant
y_corr = x1 + np.random.randn(n_samples) * 0.1

# Fit without regularization
lr_no_reg = LinearRegression()
lr_no_reg.fit(X_corr, y_corr)

# Fit with Ridge regularization
ridge = Ridge(alpha=1.0)
ridge.fit(X_corr, y_corr)

print("Correlation between features:", np.corrcoef(X_corr.T)[0, 1])
print("\nWithout regularization:")
print(f"  Coefficients: {lr_no_reg.coef_}")
print(f"  Interpretation: unstable, coefficients flip signs and magnitudes")
print("\nWith Ridge regularization (alpha=1.0):")
print(f"  Coefficients: {ridge.coef_}")
print(f"  Interpretation: stable, similar magnitudes")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Without regularization
axes[0].barh(['x1', 'x2'], lr_no_reg.coef_, color=['blue', 'orange'])
axes[0].set_title('Without Regularization (Unstable)')
axes[0].set_xlabel('Coefficient Value')
axes[0].axvline(0, color='k', linestyle='--', alpha=0.3)

# With Ridge
axes[1].barh(['x1', 'x2'], ridge.coef_, color=['blue', 'orange'])
axes[1].set_title('With Ridge Regularization (Stable)')
axes[1].set_xlabel('Coefficient Value')
axes[1].axvline(0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nWarning: Correlation does not imply causation, and unstable coefficients")
print("do not indicate causal relationships even if they were stable.")

## 5. Baseline Diagnostic

We fit a linear model to the make_circles dataset, which has a nonlinear structure. By examining residuals, we can diagnose that the model is missing important structure.

In [ ]:
# Generate make_circles data
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, random_state=42)

# Fit linear logistic regression (will fail)
lr_circles = LogisticRegression(max_iter=1000, random_state=42)
lr_circles.fit(X_circles, y_circles)

# Get predictions and residuals
y_pred = lr_circles.predict(X_circles)
residuals = (y_circles != y_pred).astype(int)

# Compute angle from center for diagnostic plot
angles = np.arctan2(X_circles[:, 1], X_circles[:, 0])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Decision boundary
h = 0.02
x_min, x_max = X_circles[:, 0].min() - 0.1, X_circles[:, 0].max() + 0.1
y_min, y_max = X_circles[:, 1].min() - 0.1, X_circles[:, 1].max() + 0.1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = lr_circles.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

axes[0].contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
axes[0].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap=plt.cm.RdYlBu, edgecolors='k', s=30)
axes[0].set_title('Linear Model Decision Boundary (Fails on Circles)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Residual pattern by angle
axes[1].scatter(angles, residuals, alpha=0.5, s=30)
axes[1].set_xlabel('Angle from Center')
axes[1].set_ylabel('Misclassification (1=error, 0=correct)')
axes[1].set_title('Residuals by Angle: Missing Structure')
axes[1].set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()

accuracy = lr_circles.score(X_circles, y_circles)
print(f"Linear model accuracy: {accuracy:.3f}")
print(f"\nDiagnosis: The residual pattern reveals systematic errors.")
print(f"The model fails because the data has a radial (distance-based) structure,")
print(f"not a linear one. We need a nonlinear model or engineered features.")

## What to Notice

1. **Linear regression recovers true coefficients** when data is generated from a linear model and noise is small.
2. **Feature engineering (polynomial features) can transform nonlinear problems** into ones that linear models can solve.
3. **Regularization induces sparsity**: Lasso drives small coefficients to exactly zero, implementing feature selection.
4. **Correlated features make coefficients unstable** and unreliable for interpretation. Ridge regularization helps, but causation still requires domain knowledge and experiments.
5. **Residual patterns diagnose model mismatch**: Systematic errors in residuals (like the circular pattern in make_circles) reveal that we need a different model family or features.

Linear models are powerful and interpretable, but they require the right features or a linear underlying structure. Always visualize data and residuals before trusting coefficients.